In [0]:
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
#import torch_directml
import torch.nn as nn

import torch.nn as nn
    
class MelDiscriminator(nn.Module):
    def __init__(self, features_dim=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(1, features_dim, 4, 2, 1, bias=False),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(features_dim, features_dim*2, 4, 2, 1, bias=False),
            nn.InstanceNorm2d(features_dim*2),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(features_dim*2, features_dim*4, 4, 2, 1, bias=False),
            nn.InstanceNorm2d(features_dim*4),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(features_dim*4, features_dim*8, 4, 2, 1, bias=False),
            nn.InstanceNorm2d(features_dim*8),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(features_dim*8, features_dim*16, 4, 2, 1, bias=False),
            nn.InstanceNorm2d(features_dim*16),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(features_dim*16, 1, kernel_size=(2, 16), stride=1, padding=0, bias=False),
            # No Sigmoid — WGAN needs raw scores
        )

    def forward(self, x):
        return self.net(x)
    
class MelGenerator(nn.Module):
    """Generator producing (1, 64, 512) mel spectrograms from latent vector."""
    def __init__(self, latent_dim=100, features_dim=64):
        super().__init__()
        self.net = nn.Sequential(
            # Input: (latent_dim, 1, 1)
            # → (features_dim*16, 2, 16)
            nn.ConvTranspose2d(latent_dim, features_dim*16, kernel_size=(2, 16), stride=1, padding=0, bias=False),
            nn.BatchNorm2d(features_dim*16),
            nn.ReLU(True),
            # → (features_dim*8, 4, 32)
            nn.ConvTranspose2d(features_dim*16, features_dim*8, 4, 2, 1, bias=False),
            nn.BatchNorm2d(features_dim*8),
            nn.ReLU(True),
            # → (features_dim*4, 8, 64)
            nn.ConvTranspose2d(features_dim*8, features_dim*4, 4, 2, 1, bias=False),
            nn.BatchNorm2d(features_dim*4),
            nn.ReLU(True),
            # → (features_dim*2, 16, 128)
            nn.ConvTranspose2d(features_dim*4, features_dim*2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(features_dim*2),
            nn.ReLU(True),
            # → (features_dim, 32, 256)
            nn.ConvTranspose2d(features_dim*2, features_dim, 4, 2, 1, bias=False),
            nn.BatchNorm2d(features_dim),
            nn.ReLU(True),
            # → (1, 64, 512)
            nn.ConvTranspose2d(features_dim, 1, 4, 2, 1, bias=False),
            nn.Tanh()
        )

    def forward(self, x):
        return self.net(x)
    
class MelChunkDataset(Dataset):
    def __init__(self, npy_path):
        data = np.load(npy_path)
        self.data = torch.from_numpy(data)

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        chunk = self.data[idx]                          # (1, 64, 431)
        chunk = torch.nn.functional.pad(chunk, (0, 81)) # pad to (1, 64, 512)
        return chunk
      

dataset = MelChunkDataset("mel_electronic_dataset.npy")
loader = DataLoader(dataset, batch_size=128, shuffle=True, num_workers=0, pin_memory=True)

print(f"Dataset size: {len(dataset)} chunks")
print(f"Batch shape: {next(iter(loader)).shape}")  # should be (128, 1, 64, 256)


def gradient_penalty(disc, real, fake, device):
    B = real.size(0)
    alpha = torch.rand(B, 1, 1, 1, device=device)
    interpolated = (alpha * real + (1 - alpha) * fake).requires_grad_(True)
    d_interp = disc(interpolated)
    gradients = torch.autograd.grad(
        outputs=d_interp, inputs=interpolated,
        grad_outputs=torch.ones_like(d_interp),
        create_graph=True, retain_graph=True
    )[0]
    gradients = gradients.view(B, -1)
    gp = ((gradients.norm(2, dim=1) - 1) ** 2).mean()
    return gp

def get_device():
    """Select device where to perform the computations."""
    #if torch_directml.is_available():
    #    return torch_directml.device()
    if torch.cuda.is_available():
        return torch.device("cuda:0")
    elif torch.backends.mps.is_available():
        return torch.device("mps")
    else:
        return torch.device("cpu")

device = get_device()

print(f"Selected device: {device}")


Using TensorFlow backend.


In [ ]:
import deeplay as dl
import time
from datetime import timedelta


latent_dim = 100
features_dim = 64

gen = MelGenerator(latent_dim=latent_dim, features_dim=features_dim).to(device)

disc = MelDiscriminator(features_dim=features_dim).to(device)


#criterion = nn.BCEWithLogitsLoss()
optim_gen = torch.optim.RMSprop(gen.parameters(), lr=0.00005)
optim_disc = torch.optim.RMSprop(disc.parameters(), lr=0.00005)

LAMBDA_GP = 10
CRITIC_STEPS = 5  #

epochs = 150
batch_size = 128
num_batches = len(loader)
gen_losses_avg, disc_losses_avg = [], []
fix_latent_vector = torch.randn(30, latent_dim, 1, 1).to(device)
for epoch in range(epochs):
    gen.train(), disc.train()

    print("\n" + f"Epoch {epoch + 1 }/{epochs}" + "\n" + "-" * 10)
    start_time = time.time()

    running_gen_loss, running_disc_loss = 0.0, 0.0

    for batch_idx, real_images in enumerate(loader):
        real_images = real_images.to(device)
        B = real_images.size(0)

        # --- Discriminator (critic) ---
        for _ in range(CRITIC_STEPS):
            noise = torch.randn(B, latent_dim, 1, 1, device=device)
            fake_images = gen(noise).detach()

            real_out = disc(real_images).reshape(-1)
            fake_out = disc(fake_images).reshape(-1)
            gp = gradient_penalty(disc, real_images, fake_images, device)

            disc_loss = -(real_out.mean() - fake_out.mean()) + LAMBDA_GP * gp
            optim_disc.zero_grad()
            disc_loss.backward()
            optim_disc.step()

        # --- Generator ---
        noise = torch.randn(B, latent_dim, 1, 1, device=device)
        fake_images = gen(noise)
        gen_loss = -disc(fake_images).reshape(-1).mean()
        optim_gen.zero_grad()
        gen_loss.backward()
        optim_gen.step()

        running_gen_loss += gen_loss.item()
        running_disc_loss += disc_loss.item()
    


    gen_losses_avg.append(running_gen_loss / num_batches)
    disc_losses_avg.append(running_disc_loss / num_batches)
    end_time = time.time()

    print("-" * 10 + "\n" + f"Epoch {epoch + 1}/{epochs}: "
          f"Generator Loss: {gen_losses_avg[-1]:.4f}, "
          f"Discriminator Loss: {disc_losses_avg[-1]:.4f}, "
          f"Time taken: {timedelta(seconds=end_time - start_time)}")
    torch.save({'epoch' : epoch + 1,
                'gen_state_dict' :gen.state_dict(),
                'gen_losses' : gen_losses_avg,
                'disc_losses' : disc_losses_avg,
    }, f"gan_checkpoint_epoch{epoch+1}.pt")
    '''
    gen.eval(), disc.eval()
    fake_images = gen(fix_latent_vector).detach().cpu().numpy()

    fig, axs = plt.subplots(3, 10, figsize=(20, 6))
    for i, ax in enumerate(axs.ravel()):
        ax.imshow(fake_images[i][0], cmap="gray")
        ax.axis("off")
    plt.tight_layout()
    plt.show()
    plt.close(fig)
    '''